# 📊 Clase 1: LangGraph & LangSmith

## Bienvenido a la Semana 3, Clase 1

En esta clase aprenderás:
- ✅ LangGraph: Flujos complejos con grafos
- ✅ Nodes, Edges, Conditional Edges
- ✅ State Management
- ✅ Memory y persistencia
- ✅ LangSmith: Debugging y monitoreo
- ✅ Function calling avanzado

---

In [ ]:
!pip install langgraph langsmith langchain-openai -q

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
import operator

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

load_dotenv()

# Configurar LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "taller-ia-semana3"

llm = ChatOpenAI(model="gpt-4", temperature=0.7)
print("✅ LangGraph y LangSmith configurados")

## 🔄 Parte 1: ¿Por qué LangGraph?

### LangChain vs LangGraph

**LangChain (Chains)**:
```
A → B → C → D
(Flujo lineal)
```

**LangGraph (Graphs)**:
```
      A
     / \
    B   C
     \ /
      D
(Flujo con decisiones)
```

### Casos de Uso de LangGraph

- ✅ Flujos con decisiones condicionales
- ✅ Loops y ciclos
- ✅ Estado compartido entre pasos
- ✅ Sistemas multi-agente
- ✅ Workflows complejos

## 📊 Parte 2: Conceptos Básicos de LangGraph

### 1. State (Estado)

El estado es la información que se comparte entre nodos.

In [ ]:
# Definir el estado
class GraphState(TypedDict):
    """Estado del grafo."""
    messages: Annotated[list, operator.add]  # Lista que se va agregando
    current_step: str
    result: str

print("✅ Estado definido")
print("Campos: messages, current_step, result")

### 2. Nodes (Nodos)

Los nodos son funciones que procesan el estado.

In [ ]:
def node_inicio(state: GraphState) -> GraphState:
    """Nodo de inicio."""
    print("🟢 Ejecutando: Nodo Inicio")
    state["messages"].append("Inicio del proceso")
    state["current_step"] = "inicio"
    return state

def node_proceso(state: GraphState) -> GraphState:
    """Nodo de procesamiento."""
    print("🔵 Ejecutando: Nodo Proceso")
    state["messages"].append("Procesando datos")
    state["current_step"] = "proceso"
    return state

def node_final(state: GraphState) -> GraphState:
    """Nodo final."""
    print("🔴 Ejecutando: Nodo Final")
    state["messages"].append("Proceso completado")
    state["result"] = "Éxito"
    return state

print("✅ Nodos definidos")

### 3. Crear el Grafo

In [ ]:
# Crear grafo
workflow = StateGraph(GraphState)

# Agregar nodos
workflow.add_node("inicio", node_inicio)
workflow.add_node("proceso", node_proceso)
workflow.add_node("final", node_final)

# Definir flujo (edges)
workflow.set_entry_point("inicio")  # Punto de entrada
workflow.add_edge("inicio", "proceso")  # inicio → proceso
workflow.add_edge("proceso", "final")  # proceso → final
workflow.add_edge("final", END)  # final → END

# Compilar
app = workflow.compile()

print("✅ Grafo creado y compilado")

In [ ]:
# Ejecutar el grafo
initial_state = {
    "messages": [],
    "current_step": "",
    "result": ""
}

print("🚀 Ejecutando grafo...\n")
final_state = app.invoke(initial_state)

print("\n📊 Estado Final:")
print(f"Mensajes: {final_state['messages']}")
print(f"Resultado: {final_state['result']}")

## 🔀 Parte 3: Conditional Edges (Decisiones)

Los **conditional edges** permiten que el grafo tome decisiones.

In [ ]:
# Estado con decisión
class DecisionState(TypedDict):
    input_text: str
    sentiment: str
    result: str

# Nodos
def analyze_sentiment(state: DecisionState) -> DecisionState:
    """Analiza el sentimiento del texto."""
    text = state["input_text"].lower()
    
    if any(word in text for word in ["excelente", "genial", "increíble"]):
        state["sentiment"] = "positivo"
    elif any(word in text for word in ["malo", "terrible", "horrible"]):
        state["sentiment"] = "negativo"
    else:
        state["sentiment"] = "neutral"
    
    print(f"📊 Sentimiento detectado: {state['sentiment']}")
    return state

def handle_positive(state: DecisionState) -> DecisionState:
    """Maneja sentimiento positivo."""
    print("😊 Procesando sentimiento positivo")
    state["result"] = "¡Gracias por tu feedback positivo!"
    return state

def handle_negative(state: DecisionState) -> DecisionState:
    """Maneja sentimiento negativo."""
    print("😔 Procesando sentimiento negativo")
    state["result"] = "Lamentamos tu experiencia. ¿Cómo podemos mejorar?"
    return state

def handle_neutral(state: DecisionState) -> DecisionState:
    """Maneja sentimiento neutral."""
    print("😐 Procesando sentimiento neutral")
    state["result"] = "Gracias por tu feedback."
    return state

# Función de decisión
def decide_next_step(state: DecisionState) -> str:
    """Decide qué nodo ejecutar según el sentimiento."""
    return state["sentiment"]

print("✅ Nodos de decisión definidos")

In [ ]:
# Crear grafo con decisiones
decision_workflow = StateGraph(DecisionState)

# Agregar nodos
decision_workflow.add_node("analyze", analyze_sentiment)
decision_workflow.add_node("positivo", handle_positive)
decision_workflow.add_node("negativo", handle_negative)
decision_workflow.add_node("neutral", handle_neutral)

# Punto de entrada
decision_workflow.set_entry_point("analyze")

# Conditional edge
decision_workflow.add_conditional_edges(
    "analyze",
    decide_next_step,
    {
        "positivo": "positivo",
        "negativo": "negativo",
        "neutral": "neutral"
    }
)

# Todos terminan en END
decision_workflow.add_edge("positivo", END)
decision_workflow.add_edge("negativo", END)
decision_workflow.add_edge("neutral", END)

# Compilar
decision_app = decision_workflow.compile()

print("✅ Grafo con decisiones creado")

In [ ]:
# Probar con diferentes textos
textos_prueba = [
    "Este producto es excelente, me encanta",
    "Terrible experiencia, muy malo",
    "Es un producto normal"
]

for texto in textos_prueba:
    print(f"\n{'='*80}")
    print(f"📝 Texto: '{texto}'")
    print("="*80)
    
    result = decision_app.invoke({"input_text": texto, "sentiment": "", "result": ""})
    print(f"\n✅ Resultado: {result['result']}")

## 🔁 Parte 4: Loops y Ciclos

In [ ]:
# Estado con contador
class LoopState(TypedDict):
    counter: int
    max_iterations: int
    results: Annotated[list, operator.add]

def process_iteration(state: LoopState) -> LoopState:
    """Procesa una iteración."""
    state["counter"] += 1
    state["results"].append(f"Iteración {state['counter']}")
    print(f"🔄 Iteración {state['counter']}/{state['max_iterations']}")
    return state

def should_continue(state: LoopState) -> str:
    """Decide si continuar o terminar."""
    if state["counter"] < state["max_iterations"]:
        return "continue"
    return "end"

# Crear grafo con loop
loop_workflow = StateGraph(LoopState)
loop_workflow.add_node("process", process_iteration)
loop_workflow.set_entry_point("process")

# Loop condicional
loop_workflow.add_conditional_edges(
    "process",
    should_continue,
    {
        "continue": "process",  # Vuelve a process
        "end": END
    }
)

loop_app = loop_workflow.compile()

print("✅ Grafo con loop creado")

In [ ]:
# Ejecutar loop
print("🔁 Ejecutando loop...\n")
result = loop_app.invoke({"counter": 0, "max_iterations": 5, "results": []})

print(f"\n✅ Completado: {result['results']}")

## 🤖 Parte 5: Ejemplo Práctico - Sistema de Investigación

In [ ]:
# Sistema que investiga un tema paso a paso
class ResearchState(TypedDict):
    topic: str
    outline: str
    research: str
    draft: str
    final_report: str

def create_outline(state: ResearchState) -> ResearchState:
    """Crea un outline del tema."""
    print("📋 Creando outline...")
    prompt = ChatPromptTemplate.from_template(
        "Crea un outline de 3 puntos para investigar sobre: {topic}"
    )
    chain = prompt | llm | StrOutputParser()
    state["outline"] = chain.invoke({"topic": state["topic"]})
    return state

def do_research(state: ResearchState) -> ResearchState:
    """Investiga cada punto del outline."""
    print("🔍 Investigando...")
    prompt = ChatPromptTemplate.from_template(
        "Investiga estos puntos sobre {topic}:\n{outline}\n\nProporciona información detallada."
    )
    chain = prompt | llm | StrOutputParser()
    state["research"] = chain.invoke({"topic": state["topic"], "outline": state["outline"]})
    return state

def write_draft(state: ResearchState) -> ResearchState:
    """Escribe un borrador."""
    print("✍️ Escribiendo borrador...")
    prompt = ChatPromptTemplate.from_template(
        "Escribe un artículo conciso basado en esta investigación:\n{research}"
    )
    chain = prompt | llm | StrOutputParser()
    state["draft"] = chain.invoke({"research": state["research"]})
    return state

def finalize_report(state: ResearchState) -> ResearchState:
    """Finaliza el reporte."""
    print("✅ Finalizando reporte...")
    state["final_report"] = f"""# {state['topic']}

{state['draft']}
"""
    return state

print("✅ Nodos de investigación definidos")

In [ ]:
# Crear grafo de investigación
research_workflow = StateGraph(ResearchState)

research_workflow.add_node("outline", create_outline)
research_workflow.add_node("research", do_research)
research_workflow.add_node("draft", write_draft)
research_workflow.add_node("finalize", finalize_report)

research_workflow.set_entry_point("outline")
research_workflow.add_edge("outline", "research")
research_workflow.add_edge("research", "draft")
research_workflow.add_edge("draft", "finalize")
research_workflow.add_edge("finalize", END)

research_app = research_workflow.compile()

print("✅ Sistema de investigación creado")

In [ ]:
# Ejecutar investigación
topic = "Inteligencia Artificial en la educación"

print(f"🚀 Investigando: {topic}\n")
print("="*80)

result = research_app.invoke({
    "topic": topic,
    "outline": "",
    "research": "",
    "draft": "",
    "final_report": ""
})

print("\n📄 REPORTE FINAL:")
print("="*80)
print(result["final_report"])

## 📊 Parte 6: LangSmith - Debugging y Monitoreo

**LangSmith** te permite:
- 🔍 Ver cada paso del grafo
- ⏱️ Medir tiempos de ejecución
- 💰 Analizar costos
- 🐛 Debuggear problemas
- 📊 Evaluar calidad

### Acceder a LangSmith

1. Ve a https://smith.langchain.com/
2. Busca tu proyecto: `taller-ia-semana3`
3. Ve los traces de las ejecuciones

### Características

- **Traces**: Ve cada llamada al LLM
- **Latency**: Tiempo de cada paso
- **Tokens**: Consumo de tokens
- **Errors**: Errores y excepciones
- **Feedback**: Evalúa respuestas

## 💡 Ejercicios Prácticos

In [ ]:
# Ejercicio 1: Crea un grafo con validación
# El grafo debe:
# 1. Recibir un email
# 2. Validar si es válido
# 3. Si es válido → enviar confirmación
# 4. Si no es válido → pedir corrección

# 👉 Tu código aquí
import re

class EmailState(TypedDict):
    email: str
    is_valid: bool
    message: str

def validate_email(state: EmailState) -> EmailState:
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    state["is_valid"] = bool(re.match(pattern, state["email"]))
    return state

def send_confirmation(state: EmailState) -> EmailState:
    state["message"] = f"✅ Email {state['email']} registrado exitosamente"
    return state

def request_correction(state: EmailState) -> EmailState:
    state["message"] = f"❌ Email {state['email']} no es válido. Por favor corrige."
    return state

# Crear y probar
print("Ejercicio 1: Sistema de validación de email")

## 🎓 Resumen

### Conceptos Clave

1. **LangGraph**: Flujos complejos con grafos
2. **State**: Información compartida
3. **Nodes**: Funciones que procesan estado
4. **Edges**: Conexiones entre nodos
5. **Conditional Edges**: Decisiones dinámicas
6. **Loops**: Ciclos y repeticiones
7. **LangSmith**: Debugging y monitoreo

### Cuándo Usar LangGraph

- ✅ Flujos con decisiones
- ✅ Loops y ciclos
- ✅ Estado compartido complejo
- ✅ Sistemas multi-agente
- ❌ Flujos simples lineales (usa Chains)

### Próxima Clase

En **Clase 2** aprenderemos:
- 🤖 Sistemas Multi-Agente
- 🏗️ Arquitecturas avanzadas
- 🔄 Coordinación entre agentes

---

**¡Excelente trabajo! 🚀**